In [1]:
import geopandas as gpd
from pathlib import Path

In [2]:
data_path = Path.cwd() / 'data'
pasy = gpd.read_file(str(data_path / 'pl_pasy_ukształtowania.geojson'))
miasta = gpd.read_file(str(data_path / "pl_miasta_wojewódzkie.geojson"))
rzeki = gpd.read_file(str(data_path / "pl_rzeki.geojson"))

"""
1. Dla warstwy z pasami ukształtowania policz ich powierzchnie w m2, korzystając z układu epsg:2180

2. Stwórz obiekt Bounding Box dla warstwy z miastami wojewódzkimi, a następnie oblicz jego obwód

3. Dla warstwy z rzekami stwórz nową kolumnę w której wyliczysz liczbę linii (LineString) znajdujących się wewnątrz. Analogicznie stwórz następnie drugą kolumnę w której wyliczysz łączną liczbę punktów składających się na geometrię
"""

'\n1. Dla warstwy z pasami ukształtowania policz ich powierzchnie w m2, korzystając z układu epsg:2180\n\n2. Stwórz obiekt Bounding Box dla warstwy z miastami wojewódzkimi, a następnie oblicz jego obwód\n\n3. Dla warstwy z rzekami stwórz nową kolumnę w której wyliczysz liczbę linii (LineString) znajdujących się wewnątrz. Analogicznie stwórz następnie drugą kolumnę w której wyliczysz łączną liczbę punktów składających się na geometrię\n'

In [3]:
#1
pasy.to_crs("EPSG:2180", inplace=True)
pasy_area = sum(pasy.area)
print(f"powierzchnia wynosi: {pasy_area:.2f} m2")

powierzchnia wynosi: 312178911190.68 m2


In [4]:
#2
bbox_miasta = miasta.to_crs("EPSG:2180").dissolve().envelope
obwod = bbox_miasta.length.values[0]
print(f"Obwód bbox wynosi: {obwod / 1000:.2f} km")

Obwód bbox wynosi: 2097.81 km


In [6]:
#3
rzeki['liczba_linii'] = rzeki.geometry.apply(
    lambda g: len(g.geoms) if g.geom_type == 'MultiLineString'
    else (1 if g.geom_type == 'LineString' else 0)
)

def count_vertices(geom):
    if geom is None or geom.is_empty:
        return 0
    elif geom.geom_type == 'LineString':
        return len(geom.coords)
    elif geom.geom_type == 'MultiLineString':
        return sum(len(line.coords) for line in geom.geoms)
    return 0

rzeki['liczba_punktow'] = rzeki.geometry.apply(count_vertices)

In [7]:
rzeki['liczba_punktow']

0       2261
1       2462
2       2761
3       1870
4       1073
       ...  
198     3916
199    10741
200    14448
201     3270
202      343
Name: liczba_punktow, Length: 203, dtype: int64